In [ ]:
# [1] Load OLD dataset (Mendeley): exploration only, no diagnosis column

df.columns = df.columns.str.strip()
df.info()
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 373 entries, 0 to 372
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   S. No.    364 non-null    float64
 1   Age       364 non-null    float64
 2   Sex       364 non-null    float64
 3   RBC       364 non-null    str    
 4   PCV       364 non-null    str    
 5   MCV       364 non-null    str    
 6   MCH       364 non-null    str    
 7   MCHC      364 non-null    float64
 8   RDW       364 non-null    str    
 9   TLC       364 non-null    str    
 10  PLT /mm3  364 non-null    str    
 11  HGB       364 non-null    str    
dtypes: float64(4), str(8)
memory usage: 35.1 KB
           S. No.         Age         Sex        MCHC
count  364.000000  364.000000  364.000000  364.000000
mean   182.500000   44.917582    0.442308   32.053407
std    105.221988   18.780854    0.497344    2.801790
min      1.000000   11.000000    0.000000   23.600000
25%     91.750000   28.000000    0.000000

In [ ]:
# [2] Load & inspect NEW dataset (Anemia Types Classification)

df2 = pd.read_csv('../data/diagnosed_cbc_data_v4.csv')
df2.columns = df2.columns.str.strip()
print(df2.shape)
print(df2.columns.tolist())
print(df2['Diagnosis'].value_counts())
print(df2.duplicated().sum())

(1281, 15)
['WBC', 'LYMp', 'NEUTp', 'LYMn', 'NEUTn', 'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT', 'PDW', 'PCT', 'Diagnosis']
Diagnosis
Healthy                           336
Normocytic hypochromic anemia     279
Normocytic normochromic anemia    269
Iron deficiency anemia            189
Thrombocytopenia                   73
Other microcytic anemia            59
Leukemia                           47
Macrocytic anemia                  18
Leukemia with thrombocytopenia     11
Name: count, dtype: int64
49


In [ ]:
# [3] Clean data: drop duplicates + remove non-anemia classes (Option A)

df2 = df2.drop_duplicates().reset_index(drop=True)

remove = ['Thrombocytopenia', 'Leukemia', 'Leukemia with thrombocytopenia']
df2 = df2[~df2['Diagnosis'].isin(remove)].reset_index(drop=True)

print(df2.shape)
print(df2['Diagnosis'].value_counts())
df2.info()
print(df2.isnull().sum())

(1105, 15)
Diagnosis
Healthy                           323
Normocytic hypochromic anemia     271
Normocytic normochromic anemia    255
Iron deficiency anemia            184
Other microcytic anemia            56
Macrocytic anemia                  16
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 1105 entries, 0 to 1104
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   WBC        1105 non-null   float64
 1   LYMp       1105 non-null   float64
 2   NEUTp      1105 non-null   float64
 3   LYMn       1105 non-null   float64
 4   NEUTn      1105 non-null   float64
 5   RBC        1105 non-null   float64
 6   HGB        1105 non-null   float64
 7   HCT        1105 non-null   float64
 8   MCV        1105 non-null   float64
 9   MCH        1105 non-null   float64
 10  MCHC       1105 non-null   float64
 11  PLT        1105 non-null   float64
 12  PDW        1105 non-null   float64
 13  PCT        1105 non-null

In [ ]:
# [4] Split data into train/test (80/20, stratified)

from sklearn.model_selection import train_test_split

X = df2.drop(columns=['Diagnosis'])
y = df2['Diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_test.value_counts())

(884, 14) (221, 14)
Diagnosis
Healthy                           65
Normocytic hypochromic anemia     54
Normocytic normochromic anemia    51
Iron deficiency anemia            37
Other microcytic anemia           11
Macrocytic anemia                  3
Name: count, dtype: int64


In [14]:
# [5] Baseline model: train Random Forest and measure train vs test accuracy
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

Train accuracy: 1.0
Test accuracy: 0.9864253393665159
                                precision    recall  f1-score   support

                       Healthy       1.00      1.00      1.00        65
        Iron deficiency anemia       0.97      1.00      0.99        37
             Macrocytic anemia       1.00      0.33      0.50         3
 Normocytic hypochromic anemia       1.00      1.00      1.00        54
Normocytic normochromic anemia       0.96      1.00      0.98        51
       Other microcytic anemia       1.00      0.91      0.95        11

                      accuracy                           0.99       221
                     macro avg       0.99      0.87      0.90       221
                  weighted avg       0.99      0.99      0.98       221



In [18]:
# [6b] Display setting: show the confusion matrix on one wide table
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 20)

In [20]:
# [6] Evaluate: confusion matrix to see exactly which classes get mixed up
import pandas as pd
from sklearn.metrics import confusion_matrix

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
print(pd.DataFrame(cm, index=labels, columns=labels))

                                Healthy  Iron deficiency anemia  Macrocytic anemia  Normocytic hypochromic anemia  Normocytic normochromic anemia  Other microcytic anemia
Healthy                              65                       0                  0                              0                               0                        0
Iron deficiency anemia                0                      37                  0                              0                               0                        0
Macrocytic anemia                     0                       0                  1                              0                               2                        0
Normocytic hypochromic anemia         0                       0                  0                             54                               0                        0
Normocytic normochromic anemia        0                       0                  0                              0                              51

In [21]:
# [7] Improve: Random Forest with class_weight='balanced' to give rare classes more importance
model_bal = RandomForestClassifier(random_state=42, class_weight='balanced')
model_bal.fit(X_train, y_train)

print("Test accuracy:", model_bal.score(X_test, y_test))
y_pred_bal = model_bal.predict(X_test)
print(classification_report(y_test, y_pred_bal))

Test accuracy: 0.9864253393665159
                                precision    recall  f1-score   support

                       Healthy       0.98      1.00      0.99        65
        Iron deficiency anemia       0.97      1.00      0.99        37
             Macrocytic anemia       1.00      0.67      0.80         3
 Normocytic hypochromic anemia       1.00      1.00      1.00        54
Normocytic normochromic anemia       0.98      0.98      0.98        51
       Other microcytic anemia       1.00      0.91      0.95        11

                      accuracy                           0.99       221
                     macro avg       0.99      0.93      0.95       221
                  weighted avg       0.99      0.99      0.99       221



In [24]:
# [8] Evaluate: 5-fold stratified cross-validation (more reliable than a single split)
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_bal, X, y, cv=cv, scoring='recall_macro')

print(scores)
print("Mean:", scores.mean())

[0.89127117 0.88562092 0.98484848 0.99673203 1.        ]
Mean: 0.9516945187165774


In [26]:
# [9] Save final model: train on ALL data, then store it in a file for the app
import os
import joblib

os.makedirs('../models', exist_ok=True)

final_model = RandomForestClassifier(random_state=42, class_weight='balanced')
final_model.fit(X, y)

joblib.dump(
    {'model': final_model, 'features': X.columns.tolist()},
    '../models/anemia_model.joblib'
)
print("Saved. Features:", X.columns.tolist())

Saved. Features: ['WBC', 'LYMp', 'NEUTp', 'LYMn', 'NEUTn', 'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT', 'PDW', 'PCT']


In [27]:
# [10] Severity rules: WHO hemoglobin cutoffs for adults (no ML, just rules)
def get_severity(hgb, sex):
    """hgb in g/dL, sex is 'male' or 'female'. Adults only, not pregnant."""
    normal_limit = 13.0 if sex == 'male' else 12.0

    if hgb >= normal_limit:
        return 'No anemia'
    elif hgb >= 11.0:
        return 'Mild'
    elif hgb >= 8.0:
        return 'Moderate'
    else:
        return 'Severe'

print(get_severity(12.5, 'male'))
print(get_severity(12.5, 'female'))
print(get_severity(9.0, 'female'))
print(get_severity(7.0, 'male'))

Mild
No anemia
Moderate
Severe


In [28]:
# [11] Advice texts: general educational tips per diagnosis (NOT medical advice)
ADVICE = {
    'Healthy': 'القيم في المعدل الطبيعي. حافظ على غذاء متوازن.',
    'Iron deficiency anemia': 'غالباً نقص حديد. الأطعمة الغنية بالحديد (لحوم، كبدة، عدس، سبانخ) مع مصدر فيتامين C بتساعد. ما تاخدش مكملات حديد قبل استشارة الطبيب.',
    'Macrocytic anemia': 'غالباً نقص B12 أو فولات. الأطعمة الغنية بيهم (لحوم، بيض، ألبان، ورقيات) مهمة. الطبيب هو اللي يحدد السبب والعلاج.',
    'Normocytic hypochromic anemia': 'ممكن يكون نقص حديد في مرحلة بدري أو مرض مزمن. محتاج تحاليل إضافية (فيريتين وحديد) وفحص طبي.',
    'Normocytic normochromic anemia': 'ممكن يكون مرتبط بأمراض مزمنة أو فقدان دم. محتاج تقييم طبي لتحديد السبب.',
    'Other microcytic anemia': 'كرات الدم صغيرة. ممكن يكون ثلاسيميا أو نقص حديد. محتاج تحاليل إضافية قبل أي مكملات.',
}
DISCLAIMER = 'للتثقيف فقط، وليس بديلاً عن استشارة الطبيب.'

In [29]:
# [12] Full pipeline: CBC values + sex -> diagnosis, severity, advice
import joblib
import pandas as pd

saved = joblib.load('../models/anemia_model.joblib')
loaded_model = saved['model']
features = saved['features']

def analyze(values, sex):
    row = pd.DataFrame([values])[features]
    diagnosis = loaded_model.predict(row)[0]
    confidence = loaded_model.predict_proba(row)[0].max()
    severity = get_severity(values['HGB'], sex)

    return {
        'diagnosis': diagnosis,
        'confidence': round(float(confidence), 2),
        'severity': severity,
        'advice': ADVICE[diagnosis],
        'disclaimer': DISCLAIMER,
    }

sample = X_test.iloc[0].to_dict()
print(analyze(sample, 'female'))

{'diagnosis': 'Normocytic normochromic anemia', 'confidence': 1.0, 'severity': 'Mild', 'advice': 'ممكن يكون مرتبط بأمراض مزمنة أو فقدان دم. محتاج تقييم طبي لتحديد السبب.', 'disclaimer': 'للتثقيف فقط، وليس بديلاً عن استشارة الطبيب.'}


In [30]:
# [13] Display: print the analysis report in a readable format
result = analyze(sample, 'female')
print("HGB used:", sample['HGB'])
for key, value in result.items():
    print(f"{key}: {value}")

HGB used: 11.8
diagnosis: Normocytic normochromic anemia
confidence: 1.0
severity: Mild
advice: ممكن يكون مرتبط بأمراض مزمنة أو فقدان دم. محتاج تقييم طبي لتحديد السبب.
disclaimer: للتثقيف فقط، وليس بديلاً عن استشارة الطبيب.


In [31]:
# [14] Compare: all 14 features vs 8 common CBC features (same cross-validation splits)
common = ['WBC', 'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT']

model_small = RandomForestClassifier(random_state=42, class_weight='balanced')
scores_small = cross_val_score(model_small, X[common], y, cv=cv, scoring='recall_macro')

print("8 features :", scores_small.round(2), "Mean:", round(scores_small.mean(), 3))
print("14 features:", scores.round(2), "Mean:", round(scores.mean(), 3))

8 features : [0.96 0.89 0.98 1.   1.  ] Mean: 0.966
14 features: [0.89 0.89 0.98 1.   1.  ] Mean: 0.952


In [32]:
# [15] Final model v2: train on ALL data with 8 common features, save, and reload for analyze()
final_small = RandomForestClassifier(random_state=42, class_weight='balanced')
final_small.fit(X[common], y)

joblib.dump(
    {'model': final_small, 'features': common},
    '../models/anemia_model_8f.joblib'
)

saved = joblib.load('../models/anemia_model_8f.joblib')
loaded_model = saved['model']
features = saved['features']

sample8 = X_test.iloc[0][common].to_dict()
print(analyze(sample8, 'female'))

{'diagnosis': 'Normocytic normochromic anemia', 'confidence': 1.0, 'severity': 'Mild', 'advice': 'ممكن يكون مرتبط بأمراض مزمنة أو فقدان دم. محتاج تقييم طبي لتحديد السبب.', 'disclaimer': 'للتثقيف فقط، وليس بديلاً عن استشارة الطبيب.'}


In [33]:
# [16] Input validation: warn if a value is outside the range the model saw in training
ranges = X[common].agg(['min', 'max'])

def check_inputs(values):
    problems = []
    for name in common:
        if name not in values:
            problems.append(f"Missing value: {name}")
            continue
        low, high = ranges.loc['min', name], ranges.loc['max', name]
        if not (low <= values[name] <= high):
            problems.append(f"{name}={values[name]} is outside the training range ({low} to {high}); check the unit")
    return problems

print(check_inputs(sample8))
bad = dict(sample8, HGB=125)
print(check_inputs(bad))

[]
['HGB=125 is outside the training range (-10.0 to 87.1); check the unit']


In [34]:
# [17] Data quality: find impossible or suspicious values in each feature
print(X[common].describe().loc[['min', 'max']].T)
print((X[common] < 0).sum())

        min     max
WBC    0.80    45.7
RBC    1.36    90.8
HGB  -10.00    87.1
HCT    2.00  3715.0
MCV  -79.30   122.1
MCH   11.40  3117.0
MCHC  11.50    92.8
PLT   11.30   660.0
WBC     0
RBC     0
HGB     1
HCT     0
MCV     1
MCH     0
MCHC    0
PLT     0
dtype: int64


In [35]:
# [18] Data quality: list rows with physiologically impossible values (inspect only, do NOT delete yet)
limits = {
    'WBC': (0.5, 100), 'RBC': (1, 10), 'HGB': (3, 25), 'HCT': (10, 70),
    'MCV': (40, 150), 'MCH': (10, 50), 'MCHC': (20, 45), 'PLT': (5, 1500),
}

bad_mask = pd.Series(False, index=df2.index)
for col, (low, high) in limits.items():
    bad_mask |= ~df2[col].between(low, high)

print("Suspicious rows:", bad_mask.sum())
print(df2.loc[bad_mask, ['Diagnosis'] + common])
print(df2.loc[bad_mask, 'Diagnosis'].value_counts())

Suspicious rows: 22
                          Diagnosis   WBC    RBC    HGB        HCT    MCV     MCH   MCHC    PLT
4     Normocytic hypochromic anemia   4.2   3.93   0.40   316.0000  80.60    23.9  29.70  236.0
5           Other microcytic anemia   6.6   3.96   8.80    29.7000  75.20    22.2  79.60  207.0
60           Iron deficiency anemia  11.8   4.58   1.20    35.1000  76.80    24.4  31.90  141.0
71    Normocytic hypochromic anemia   5.2   4.40  11.70    37.3000  84.80    26.5  61.30  210.0
76   Normocytic normochromic anemia   2.8   3.44   9.60    30.3000  88.20    27.9  61.60  133.0
176         Other microcytic anemia   6.0   3.20   4.82    12.1000  36.60    76.0  25.10   13.9
201         Other microcytic anemia   6.5  41.10  85.10    26.6000  31.30    37.2  12.40   11.8
204          Iron deficiency anemia   9.1   4.66 -10.00    34.6000  74.40    21.4  28.90  312.0
222  Normocytic normochromic anemia  10.9   4.89  14.50    14.0000  91.20    90.0  32.90   45.0
223                 

In [36]:
# [19] Clean: drop the 22 suspicious rows from [18], rebuild X and y, verify class counts
df2 = df2.loc[~bad_mask].reset_index(drop=True)

X = df2.drop(columns=['Diagnosis'])
y = df2['Diagnosis']

print(df2.shape)
print(y.value_counts())

(1083, 15)
Diagnosis
Healthy                           322
Normocytic hypochromic anemia     265
Normocytic normochromic anemia    251
Iron deficiency anemia            180
Other microcytic anemia            49
Macrocytic anemia                  16
Name: count, dtype: int64


In [37]:
# [20] Re-evaluate on cleaned data: new split + 5-fold CV with the 8 common features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model_check = RandomForestClassifier(random_state=42, class_weight='balanced')
scores_clean = cross_val_score(model_check, X[common], y, cv=cv, scoring='recall_macro')
print(scores_clean.round(2), "Mean:", round(scores_clean.mean(), 3))

model_check.fit(X_train[common], y_train)
print(classification_report(y_test, model_check.predict(X_test[common])))

[1.   0.92 1.   0.97 0.97] Mean: 0.971
                                precision    recall  f1-score   support

                       Healthy       0.98      1.00      0.99        65
        Iron deficiency anemia       0.97      1.00      0.99        36
             Macrocytic anemia       1.00      1.00      1.00         3
 Normocytic hypochromic anemia       1.00      1.00      1.00        53
Normocytic normochromic anemia       1.00      0.98      0.99        50
       Other microcytic anemia       1.00      0.90      0.95        10

                      accuracy                           0.99       217
                     macro avg       0.99      0.98      0.99       217
                  weighted avg       0.99      0.99      0.99       217



In [38]:
# [21] Display: show English name with Arabic in parentheses (translation at display time only, model unchanged)
AR_NAMES = {
    'Healthy': 'سليم',
    'Iron deficiency anemia': 'أنيميا نقص الحديد',
    'Macrocytic anemia': 'أنيميا كبيرة الكرات',
    'Normocytic hypochromic anemia': 'أنيميا عادية الحجم ناقصة الصبغة',
    'Normocytic normochromic anemia': 'أنيميا عادية الحجم والصبغة',
    'Other microcytic anemia': 'أنيميا أخرى صغيرة الكرات',
    'No anemia': 'لا توجد أنيميا',
    'Mild': 'خفيفة',
    'Moderate': 'متوسطة',
    'Severe': 'شديدة',
}

def with_arabic(name):
    return f"{name} ({AR_NAMES.get(name, name)})"

result = analyze(sample8, 'female')
print("diagnosis:", with_arabic(result['diagnosis']))
print("severity:", with_arabic(result['severity']))

diagnosis: Normocytic normochromic anemia (أنيميا عادية الحجم والصبغة)
severity: Mild (خفيفة)


In [39]:
# [22] Final model v3: recompute input ranges on clean data, retrain on ALL clean data, save with ranges
ranges = X[common].agg(['min', 'max'])
print(ranges.T)

final_clean = RandomForestClassifier(random_state=42, class_weight='balanced')
final_clean.fit(X[common], y)

joblib.dump(
    {'model': final_clean, 'features': common, 'ranges': ranges},
    '../models/anemia_model_8f_clean.joblib'
)

saved = joblib.load('../models/anemia_model_8f_clean.joblib')
loaded_model = saved['model']
features = saved['features']
ranges = saved['ranges']

sample_clean = X.iloc[0][common].to_dict()
print(check_inputs(sample_clean))
print(analyze(sample_clean, 'female'))

        min    max
WBC    0.80   45.7
RBC    1.36    6.9
HGB    3.10   16.2
HCT   10.10   49.8
MCV   55.70  122.1
MCH   11.40   41.2
MCHC  20.20   42.0
PLT   12.40  660.0
[]
{'diagnosis': 'Normocytic hypochromic anemia', 'confidence': 0.98, 'severity': 'Severe', 'advice': 'ممكن يكون نقص حديد في مرحلة بدري أو مرض مزمن. محتاج تحاليل إضافية (فيريتين وحديد) وفحص طبي.', 'disclaimer': 'للتثقيف فقط، وليس بديلاً عن استشارة الطبيب.'}
